In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")

In [ ]:
from scripts.utils.experiment_tools import SafeExperimentModule

experimenter = SafeExperimentModule("../data/sim2")

In [ ]:
def find_experiment_ids(search_filter):
    if not Path(experimenter.metafile).is_file():
        raise FileNotFoundError("Run the simulations in the README to create data/sim2/meta.json.")

    return [
        record for record in experimenter.read_meta_json()
        if all(record["inputs"].get(key) == value for key, value in search_filter.items())
    ]

In [ ]:
# Select a simulation. Use the last matching run by default.
dataset_name = "li_li_correct_continuous"
policy_class = "linear"
experiment_index = -1

experiments = find_experiment_ids({
    "dataset_name": dataset_name,
    "policy_class": policy_class,
    "correct_for_bias": True,
})
if not experiments:
    raise ValueError("No matching simulation run. Check the settings or run this simulation first.")

for i, exp in enumerate(experiments):
    print(i, exp["_id"], exp["inputs"])

In [ ]:
df = experimenter.read_experiment_results(experiments[experiment_index]['_id'])

df['model'] = df['model'].str.lower()
df = df.loc[df['model'].isin(['logistic', 'bt'])]
print(df.shape)
df.columns

In [ ]:
df.head()

In [ ]:
# processing for plot

# make the plot 
sampling_map = {
    'uniform': 'Uniform', 
    'adaptive': 'Adaptive', 
    'double_ts': 'DTS', 
    'maxent': 'MaxEnt', 
    'maxintp': 'MaxIntP'
}

model_map = {
    'logistic': 'Logistic', 
    'bt': 'BTL'
}

# 2. Map the raw values to proper names (fallback to original if not found in dict)
df['sampling_clean'] = df['sampling'].map(sampling_map).fillna(df['sampling'])
df['model_clean'] = df['model'].map(model_map).fillna(df['model'])

# 3. Create a single, formatted column for the legend (e.g., "DTS - Logistic" or "MaxEnt (BT MLE)")
# You can change the formatting string below to whatever looks best to you.
df['Legend_Label'] = df['sampling_clean'] + " (" + df['model_clean'] + ")"

# The released plots add 0.05 for display on the log scale.
df['regret'] = df['regret'] + 0.05
df = df.sort_values(by=["Legend_Label", "repeat", "round"])

# Calculate moving average grouped by the new label
df['regret_ma'] = df.groupby(["Legend_Label", "repeat"])['regret'].transform(
    lambda x: x.rolling(window=25, min_periods=1).mean()
)

df.head()

In [ ]:
# Make the plot using the new clean column
ax = sns.lineplot(data=df, x="round", y="regret_ma", hue="Legend_Label")

sns.despine()
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Round")
plt.ylabel("Regret")

# Format the legend 
# This removes the default column name title and places the legend outside the plot
plt.legend(
    title="Policy (Model)",     # Set a clean title for the legend box
    # bbox_to_anchor=(1.05, 1),   # Pushes the legend outside the right edge of the plot
    loc='best',           # Anchors the legend box
    frameon=True               # Removes the box outline for a cleaner look
)

# Ensures the saved figure doesn't crop the external legend
plt.tight_layout()
output_dir = Path("../data/figures/fig2")
output_dir.mkdir(parents=True, exist_ok=True)
plot_path = output_dir / f"{dataset_name}__{policy_class}.pdf"
plt.savefig(plot_path, metadata={"CreationDate": None})

print(plot_path)
plt.show()